# Kitchen Sink - Oh My! It's Everything!

This project is about setting up everything we can and seeing what happens!

Download this project by either:

1. Running the CrocoDash CLI command: `crocodash template --output kitchen_sink_case.ipynb --machine glade --notebook crocodash.projects.KitchenSink`, then filling in `<CESM>`, `<inputdir>` and `<casedir>`
2. Copying the file from your CrocoDash checkout: `gallery/crocodash/projects/KitchenSink.ipynb`

If you installed with CROCODILEworkspace (`./install.sh --workshop`), this project is already in your `workspace/projects/` folder, with your paths, the workshop queue and the project code filled in.

## Grid Generation
Let's make a small domain in a northerly area

In [ ]:
from pathlib import Path

from CrocoDash.grid import Grid
from CrocoDash.vgrid import VGrid
from CrocoDash.topo import Topo
from CrocoDash.case import Case


grid = Grid.from_projection(
    crs="EPSG:3338",        # Alaska Albers (std parallels 55N/65N, CM 154W)
    x_min=-1_000_000,       # meters, Albers origin at (154W, 50N)
    x_max= 1_200_000,
    y_min=  400_000,
    y_max= 1_600_000,
    resolution_m=10_000,    # 10 km
    name="alaska_10km",
)


topo = Topo(
    grid = grid,
    min_depth = 9.5, # in meters
)

topo.set_from_dataset(    
    bathymetry_path = "<GEBCO_LOWRES>",
    longitude_coordinate_name="lon",
    latitude_coordinate_name="lat",
    vertical_coordinate_name="elevation"
)

topo.depth.plot()
vgrid  = VGrid.hyperbolic(
    nk = 75, # number of vertical levels
    depth = topo.max_depth,
    ratio=20.0 # target ratio of top to bottom layer thicknesses
)


In [ ]:
# CESM case (experiment) name
casename = "ks_alaska_10km"

# CESM source root (Update this path accordingly!!!)
cesmroot ="<CESM>"

# Place where all your input files go 
inputdir = Path("<inputdir>") / casename
    
# CESM case directory
caseroot = Path("<casedir>") / casename


case = Case(
    cesmroot = cesmroot,
    caseroot = caseroot,
    inputdir = inputdir,
    ocn_grid = grid,
    ocn_vgrid = vgrid,
    ocn_topo = topo,
    project = '<PROJECT>',
    job_queue = "<JOB_QUEUE>", # PBS queue the run is submitted to
    job_wallclock_time = "<WALLCLOCK_1H>", # hh:mm:ss walltime requested for the run
    override = True,
    machine = "derecho",
    compset = "1850_DATM%JRA_SLND_CICE_MOM6%REGIONAL%MARBL-BIO_DROF%GLOFAS_SGLC_WW3" )


## Forcings
DROF is the only new component here. All we need to do is provide a mesh file to get the runoff going. All CrocoDash does is generate a mapping file for the runoff data. The CESM coupler actually passes the data in (which we specify through the compset).

In [ ]:
case.configure_forcings(
    date_range=["2020-01-01 00:00:00", "2020-01-09 00:00:00"],
    product_name="reference_ocean",
    function_name="get_reference_ocean_data",
    marbl_ic_filepath="<MARBL_IC>",
    restore_ice=True,
    cice_product_name="reference_ice",
    cice_function_name="get_reference_ice_data",
    ww3_obc_product_name="reference_waves",
    ww3_obc_function_name="get_reference_wave_spectra",
    rof_esmf_mesh_filepath="<GLOFAS_MESH>",
)


In [ ]:
case.process_forcings()


Let's see if this works! If you're interested in this setup, you can swap in real data for each product as you go along! It'll take some serious time to run!

With this many components active, I'd mess with the NTASKS/ROOTPE layout for each one to make sure everything looks reasonable, then a quick build & submit.

Feel free to iterate with process forcings! You can run it on the command line with `crocodash process --caseroot YOURCASEROOT --all` (or name just the pieces you changed, e.g. `--bc`, `--cice`; see `crocodash process -h`)!